# Visualisation — Plan your trip with Kayak
- Top 5 destinations recommandées (indicateur météo)
- Top 20 hôtels (score Booking) par ville

In [2]:
# initialisation
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from datetime import date
from sqlalchemy import create_engine
import os
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
engine = create_engine(f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

today = date.today()

## Géolocalisation des villes

In [48]:
df_cities = pd.read_sql("SELECT * FROM cities", engine)

In [16]:
fig = px.scatter_map(
    df_cities, lat="latitude", lon="longitude",
    zoom=5, map_style="carto-positron",
    hover_name="city",
    title="Géolocalisation des villes",
    height=800, width=800
)
fig.update_traces(marker=dict(size=12))
fig.show()

## Top 5 destinations — Indicateur météo

In [37]:
# chargement des 5 meilleures villes (weather_code_mean le plus bas = meilleur temps)
try:
    df_weather_cities = pd.read_sql(
        """SELECT c.city_id, c.city, c.latitude, c.longitude,
                wc.weather_code_mean, wc.temperature_min_mean,
                wc.temperature_max_mean, wc.precipitation_sum
        FROM weather_cities wc
        JOIN cities c ON wc.city_id = c.city_id
        WHERE wc.date = (SELECT MAX(date) FROM weather_cities)
        ORDER BY wc.weather_code_mean
        """,
        engine
    )
    display(df_weather_cities)
except Exception as e:
    print(f"Erreur lecture DB: {e}")
    df_weather_cities = pd.DataFrame()
finally:
    engine.dispose()

,city_id,city,latitude,longitude,weather_code_mean,temperature_min_mean,temperature_max_mean,precipitation_sum
0,12,Carcassonne,43.213036,2.349107,0.48,21.2,34.9,0.00
1,2,Marseille,43.296399,5.377789,0.52,18.7,27.3,0.00
2,16,Montauban,44.017584,1.354999,0.52,16.3,33.5,0.00
3,4,Bormes les Mimosas,43.150697,6.341928,0.55,18.1,27.5,0.00
4,18,Toulouse,43.604464,1.444243,0.60,17.4,32.9,0.00
5,10,Aigues Mortes,43.566152,4.191540,0.60,19.6,32.6,0.00
6,23,Cassis,43.214036,5.539632,0.61,18.3,26.4,0.00
7,8,Nimes,43.837425,4.360069,0.62,19.1,34.1,0.00
8,26,Avignon,43.949249,4.805901,0.62,18.5,33.9,0.00
9,25,Uzes,44.012128,4.419672,0.64,18.3,33.6,0.00


In [ ]:
# carte Top destinations avec indicateur météo
df_weather_cities['marker_size'] = 5
df_weather_cities.loc[:4, 'marker_size'] = 25

fig = px.scatter_map(
    df_weather_cities, lat="latitude", lon="longitude",
    zoom=5, map_style="carto-positron",
    color="weather_code_mean",
    color_continuous_scale="RdYlGn_r",
    size="marker_size",
    hover_name="city",
    hover_data={
        "weather_code_mean": True,
        "latitude": False,
        "longitude": False
    },
    labels={
        "weather_code_mean": "Météo (7j)",

    },
    title="Top 5 destinations — Meilleure météo sur 7 jours",
    height=800, width=800
)
fig.show()

## Top 20 hôtels par ville

In [44]:
# chargement des 20 meilleurs hôtels dans les top 5 villes (score le plus élevé)
df_hotels_enriched = pd.read_sql("SELECT * FROM hotels_enriched", engine)
top_cities = df_hotels_enriched['city'].unique()
print(top_cities)


['Avignon' 'Gorges du Verdon' 'Montauban' 'Nimes' 'Uzes']


In [47]:
# carte Top 20 hôtels colorés par ville
for city in top_cities:
    df_city = df_hotels_enriched[df_hotels_enriched['city'] == city].head(20)
 
    if df_city.empty:
        print(f"Aucun hôtel pour {city}")
        continue
 
    fig = px.scatter_map(
        df_city, lat="latitude", lon="longitude",
        zoom=12, map_style="carto-positron",
        color="score",
        color_continuous_scale="Greens",
        size="score",
        size_max=15,
        hover_name="name",
        hover_data={
            "score": True,
            "votes": True,
            "latitude": False,
            "longitude": False
        },
        labels={"score": "Score Booking", "votes": "Avis"},
        title=f"Top 20 hôtels — {city}",
        height=800, width=800
    )
    fig.show()